# Phase 2: High-Impact EDA

## Objective
Perform comprehensive exploratory data analysis focused on understanding fraud patterns, feature relationships, and generating actionable insights for modeling.

## Key Question to Answer:
1. What fetures best discriminate between fraud and legitimate transactions?
2. Are there temporal patterns in fraudulent activity?
3. Why do duplicate transactions have 10x higher fraud rates?
4. How does transaction amount relate to fraud probability?
5. Which features are most important for prediction?


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
from scipy import stats
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# Import utility functions
from utils import (
    load_fraud_data,
    quick_data_summary,
    plot_target_distribution,
    DATA_RAW,
    DATA_PROCESSED,
    PLOTS_DIR,
    setup_mlflow
)

# Display and plotting settings
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("=" * 60)
print("PHASE 2: HIGH-IMPACT EDA")
print("=" * 60)
print("\n✅ Environment Ready")
print(f"📊 DuckDB Version: {duckdb.__version__}")


In [ ]:
# Load dataset
print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = load_fraud_data()

# Quick summary
print("\n📊 Dataset Overview:")
print(f"  • Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"  • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  • Fraud transactions: {df['Class'].sum():,} ({(df['Class'].sum() / len(df) * 100):.3f}%)")
print(f"  • Legitimate transactions: {(df['Class']==0).sum():,} ({((df['Class']==0).sum() / len(df) * 100):.3f}%)")

# Store feature lists for later use
v_features = [f"V{i}" for i in range(1, 29)]
all_features = ["Time"] + v_features + ["Amount"]
target = "Class"

print(f"\n✅ Dataset loaded successfully")
print(f"📋 Feature groups defined:")
print(f"  • V features (PCA): {len(v_features)}")
print(f"  • Other features: Time, Amount")
print(f"  • Target: {target}")


In [ ]:
# Initialize DuckDB connection
print("=" * 60)
print("INITIALIZING DUCKDB")
print("=" * 60)

# Create DuchDB connection
con = duckdb.connect(":memory:")

# Register the DataFrame as a DuckDB table
con.register("fraud_data", df)

# Test query
test_query = """
    SELECT
        COUNT(*) as total_records,
        SUM(CASE WHEN "Class" = 1 THEN 1 ELSE 0 END) as fraud_count,
        SUM(CASE WHEN "Class" = 0 THEN 1 ELSE 0 END) as legitimate_count,
        ROUND(AVG("Amount"), 2) as avg_amount
    FROM fraud_data
"""

test_result = con.execute(test_query).df()
print("\n✅ DuckDB initialized successfully")
print("\n📊 Test Query Results:")
print(test_result)

print("\n💡 DuckDB Benefits:")
print("  • Fast SQL queries on pandas DataFrames")
print("  • Memory-efficient aggregations")
print("  • Familiar SQL syntax for complex analytics")
print("  • Perfect for exploratory analysis")

print("\n✅ Step 1 complete: Environment ready for EDA.")


## Step 2: Class-wise Feature Distribution Analysis

**Goal**: Compare statistical properties of features across fraud vs legitimate transactions.

**Key Metrics:**
- Mean and median differences
- Standard deviation comparison
- Distribution shapes (skewnesss, kurtosis)
- Statistical significance tests


In [ ]:
# Class-wise statistical comparison using DuckDB
print("" * 60)
print("STEP 2: CLASS-WISE FEATURE DISTRIBUTION ANALYSIS")
print("" * 60)

# Create comprehensive statistics query for all features
features_to_analyze = all_features

# Build query dynamically
stats_query = """
    SELECT
        'Class',
        COUNT(*) as count,
"""

# Add statistics for each feature
for feature in features_to_analyze:
    stats_query += f"""
        ROUND(AVG("{feature}"), 4) as {feature}_mean,
        ROUND(STDDEV("{feature}"), 4) as {feature}_std,
        ROUND(MEDIAN("{feature}"), 4) as {feature}_median,
        ROUND(MIN("{feature}"), 4) as {feature}_min,
        ROUND(MAX("{feature}"), 4) as {feature}_max,
    """

# Remove trailing comma and add GROUP BY
stats_query = stats_query.rstrip(",\n") + """
    FROM fraud_data
    GROUP BY "Class"
    ORDER BY "Class"
"""

print("\n🔍 Computing class-wise statistics for all features...")
class_stats = con.execute(stats_query).df()

print(f"\n✅ Statistics computed for {len(features_to_analyze)} features")
print(f"📊 Sample of results (Class distribution):")
print(f"  • Class 0 (Legitimate): {class_stats.loc[0, 'count']:,} transactions")
print(f"  • Class 1 (Fraud): {class_stats.loc[1, 'count']:,} transactions")


In [ ]:
# Extract mean differences for key features
print("=" * 70)
print("KEY FEATURE COMPARISONS: FRAUD VS LEGITIMATE")
print("=" * 70)

# Focus on Time, Amount and Top V features
key_features = ["Time", "Amount", "V1", "V2", "V3", "V4", "V5", "V10", "V12", "V14", "V17"]

print("\n📊 Mean Value Comparisons:\n")
print(f"{'Feature':<10} {'Legitimate':>15} {'Fraud':>15} {"Difference":>15} {'% Change':>12}")
print("-" * 72)

for feature in key_features:
    legit_mean = class_stats.loc[0, f"{feature}_mean"]
    fraud_mean = class_stats.loc[1, f"{feature}_mean"]
    diff = fraud_mean - legit_mean
    
    # Avoid division by zero
    if legit_mean != 0:
        pct_change = (diff / abs(legit_mean)) * 100
    else:
        pct_change = np.inf if diff != 0 else 0
    
    print(f"{feature:<10} {legit_mean:>15.4f} {fraud_mean:>15.4f} {diff:>15.4f} {pct_change:>12.2f}%")

print("\n💡 Interpretation:")
print("  • Large differences suggest discriminative power")
print("  • V features with big mean shifts are likely important")
print("  • Time and Amount differences reveal fraud patterns")


In [ ]:
# Calculate effect sizes for all features (Cohen's d)
print("=" * 70)
print("EFFECT SIZE ANALYSIS (COHEN'S D)")
print("=" * 70)

print("\n Computing effect sizes to quantify class separation...\n")

# Separate fraud and legitimate transactions
fraud_df = df[df["Class"] == 1]
legit_df = df[df["Class"] == 0]

# Calculate Cohen's d for all features
effects_sizes = []

for feature in all_features:
    # Get mean and stds
    mean_fraud = fraud_df[feature].mean()
    mean_legit = legit_df[feature].mean()
    std_fraud = fraud_df[feature].std()
    std_legit = legit_df[feature].std()
    
    # Pooled standard deviation
    n_fraud = len(fraud_df)
    n_legit = len(legit_df)
    pooled_std = np.sqrt(((n_fraud - 1) * std_fraud**2 + (n_legit - 1) * std_legit**2) / (n_fraud + n_legit - 2))
    
    # Cohen's d
    cohens_d = (mean_fraud - mean_legit) / pooled_std if pooled_std != 0 else 0
    
    effects_sizes.append({
        "Feature": feature,
        "Cohens_d": cohens_d,
        "Abs_Cohens_d": abs(cohens_d),
        "Mean_Fraud": mean_fraud,
        "Mean_Legit": mean_legit,
        "Effect_Interpretation": "Large" if abs(cohens_d) > 0.8 else "Medium" if abs(cohens_d) > 0.5 else "Small"
    })

effect_df = pd.DataFrame(effects_sizes).sort_values("Abs_Cohens_d", ascending=False)

print(f"📊 Top 15 Features by Effect Size (Discriminative Power):\n")
print(effect_df[["Feature", "Cohens_d", "Effect_Interpretation"]].head(15).to_string(index=False))

print("\n📊 Effect Size Interpretation:")
print("  • |d| > 0.8: Large effect (strong class separation)")
print("  • |d| 0.5 - 0.8: Medium effect (moderate class separation)")
print("  • |d| < 0.5: Small effect (weak class separation)")

# Count by effect size
large_effects = (effect_df["Abs_Cohens_d"] > 0.8).sum()
medium_effects = ((effect_df["Abs_Cohens_d"] > 0.5) & (effect_df["Abs_Cohens_d"] <= 0.8)).sum()
small_effects = (effect_df["Abs_Cohens_d"] <= 0.5).sum()

print("\n📈 Effect Size Distribution:")
print(f"  • Large effects: {large_effects} features")
print(f"  • Medium effects: {medium_effects} features")
print(f"  • Small effects: {small_effects} features")


In [ ]:
# Visualize top discriminative faetures
print("=" * 70)
print("VISUALIZING TOP DISCRIMINATIVE FEATURES")
print("=" * 70)

# Get top 6 features by effect size
top_features = effect_df.head(6)["Feature"].tolist()

print("\n📊 Creating distribution plots for top 6 features...")
print(f"  Features: {', '.join(top_features)}\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    print(feature)
    ax = axes[idx]
    
    # Get data for both classes
    legit_data = legit_df[feature]
    fraud_data = fraud_df[feature]
    
    # Plot distributions
    ax.hist(legit_data, bins=50, alpha=0.6, label="Legitimate", color="green", density=True)
    ax.hist(fraud_data, bins=50, alpha=0.6, label="Fraud", color="red", density=True)
    
    # Add mean lines
    ax.axvline(legit_data.mean(), color="darkgreen", linestyle="--", linewidth=2, label=f"Legit Mean: {legit_data.mean():.2f}")
    ax.axvline(fraud_data.mean(), color="darkred", linestyle="--", linewidth=2, label=f"Fraud Mean: {fraud_data.mean():.2f}")
    
    # Get Cohen's d for this feature
    cohens_d = effect_df[effect_df["Feature"] == feature]["Cohens_d"].values[0]
    
    ax.set_xlabel(feature, fontsize=11, fontweight="bold")
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"{feature} Distribution (Cohen's d = {cohens_d:.3f})", fontsize=12, fontweight="bold")
    ax.legend(loc="best", fontsize=9)
    ax.grid(alpha=0.3)
    
plt.tight_layout()

# Save the plot in local directory
plt.savefig(PLOTS_DIR / "distribution_plot.png")

plt.show()

print("✅ Distribution plots created")
print("\n💡 Key Observations:")
print("  • Clear esparation indicates strong predictive power")

print("✅ Distribution plots created")
print("\n💡 Key Observations from Visualization:")
print("  🔴 V17, V14, V12, V10: Extreme separation, fraud heavily concentrated around -7")
print("  🔴 Legitimate transactions tightly clustered near 0 (PCA standardization)")
print("  🔴 Fraud forms distinct, separate distributions, minimal overlap")
print("  🟢 V16, V3: Similar patterns, fraud shifted to negative values")
print("  ✅ Almost no overlap between distributions = exceptional predictive power")
print("  ✅ These features will be critical for fraud detection models")
print("\n🎯 Fraud Pattern: Fraud transactions show systematic negative shifts in V17, V14, V12, V10, V16, V3 compared to legitimate transactions")


In [ ]:
# Summary of findings
print("=" * 70)
print("STEP 2 SUMMARY: CLASS-WISE DISTRIBUTION INSIGHTS")
print("=" * 70)

# Identify top positive and negative Cohen's d
top_negative = effect_df.nsmallest(8, "Cohens_d")[["Feature", "Cohens_d", "Mean_Fraud", "Mean_Legit"]]
top_positive = effect_df.nlargest(5, "Cohens_d")[["Feature", "Cohens_d", "Mean_Fraud", "Mean_Legit"]]

print("\n🔴 Top 8 Features with Lower Values in Fraud (Negative Cohen's d):")
print(f"{'Feature':<10} {'Cohen\'s d':>12} {'Fraud Mean':>12} {'Legit Mean':>12} {'Separation':>12}")
print("-" * 70)

for idx, row in top_negative.iterrows():
    separation = "Extreme" if abs(row["Cohens_d"]) > 5 else "Very High" if abs(row["Cohens_d"]) > 3 else "High"
    print(f"{row['Feature']:<10} {row['Cohens_d']:>12.3f} {row['Mean_Fraud']:>12.2f} {row['Mean_Legit']:>12.2f} {separation:>12}")
    
print("\n🟢 Top 5 Features with Higher Values in Fraud (Positive Cohen's d):")
print(f"{'Feature':<10} {'Cohen\'s d':>12} {'Fraud Mean':>12} {'Legit Mean':>12} {'Separation':>12}")
print("-" * 70)

for idx, row in top_positive.iterrows():
    separation = "Extreme" if abs(row["Cohens_d"]) > 5 else "Very High" if abs(row["Cohens_d"]) > 3 else "High"
    print(f"{row['Feature']:<10} {row['Cohens_d']:>12.3f} {row['Mean_Fraud']:>12.2f} {row['Mean_Legit']:>12.2f} {separation:>12}")

print()
print("=" * 70)
print("KEY FINDINGS & INSIGHTS")
print("=" * 70)

print(f"""
🎯 EXCEPTIONAL DISCRIMINATIVEW POWER DISCOVERED

1. EXTREME Class Separation (|Cohen's d| > 5):
  • V17 (d=-8.32): Fraud mean -6.67 vs Legit mean 0.01
  • V14 (d=-7.64): Fraud mean -6.97 vs Legit mean 0.01
  • V12 (d=-6.50): Fraud mean -6.26 vs Legit mean 0.01
  • V10 (d=-5.35): Fraud mean -5.68 vs Legit mean 0.01
  
  📊 Pattern: Fraud transactions cluster around -6 to -7 while
      legitimate transactions stay near 0 (almost no overlap!)

2. Very High Separation (3 < |Cohen's d| < 5):
  • V16 (d=-4.83): Strong negative shift in fraud
  • V3  (d=-4.74): Strong negative shift in fraud
  • V7  (d=-4.59): Strong negative shift in fraud
  • V11 (d=+3.78): Fraud has HIGHER values (positive shift)
  • V4  (d=+3.24): Fraud has HIGHER values (positive shift)

3. Temporal Pattern:
  • Fraud occurs 14,091 seconds EARLIER on average
  • Fraud Time: 80,747s vs Legitimate: 94,838s
  • Suggests fraudsters act in first ~22 hours of observation period

4. Transaction Amount Pattern:
  • Fraud amounts 38% HIGHER: $122.21 vs $88.29
  • Cohen's d = 0.136 (small but consistent)
  • Fraudsters may target higher-value transactions

5. Overall Discriminative Power:
  • 17 features with LARGE effects (|d| > 0.8)
  • 0 features with Medium effects
  • 13 features with Small effects
  • Total: 30 features analyzed

💡 MODELING IMPLICATIONS:

✅ Exceptional Predictive Signals:
  • V17, V14, V12, V10 will be top model features
  • Nearly perfect separation means high accuracy is achievable
  • Ensemble methods should perform exceptionally well

✅ Feature Engineering Priorities:
  1. Keep V17, V14, V12, V10, V16, V3, V7 as-is (already powerful)
  2. Consider interaction terms: V17 x V14, V12 x V10
  3. Engineer temporal features from Time (hour, day patterns)
  4. Create Amount bins/thresholds (fraud prefers higher amounts)

✅ Model Selection Guidance:
  • Tree-based models will excel (clear decision boundaries)
  • Linear models may also work well (large separations)
  • Neural networks might be overkill given clear signals
  • Focus on handling class imbalance, not feature engineering

⚠️ Important Notes:
  • Extreme Cohen's d values are valid for this imbalanced dataset
  • The 577:1 class imbalance amplifies effect sizes mathematically
  • These features genuinely show exceptional discrimination
  • This is excellent news for model performance
""")


## Step 3: Temporal Pattern Analysis

**Goal**: Uncover temporal patterns in fraud activity and engineer time-based features.

**Analysis Plan:**
1. Time distribution by class (when does fraud occur?)
2. Convert Time to hours and analyze hourly patterns
3. Transaction velocity analysis
4. Time gaps between transactions
5. Fraud concentration windows

In [ ]:
# Temporal analysis setup
print("=" * 70)
print("STEP 3: TEMPORAL PATTERN ANALYSIS")
print("=" * 70)

print(f"\n⏰ Analyzing temporal patterns in fraud activity...")

# Basic time statistics by class
time_stats_query = """
    SELECT
        "Class",
        COUNT(*) as count,
        ROUND(MIN("TIME"), 2) as min_time,
        ROUND(MAX("TIME"), 2) as max_time,
        ROUND(AVG("TIME"), 2) as avg_time,
        ROUND(MEDIAN("TIME"), 2) as median_time,
        ROUND(STDDEV("TIME"), 2) as std_time
    FROM fraud_data
    GROUP BY "Class"
    ORDER BY "Class"
"""

time_stats = con.execute(time_stats_query).df()

print(f"📊 Time Statistics by Class:\n")
print(time_stats.to_string(index=False))

# Calculate time span
total_time_seconds = df["Time"].max()
total_time_hours = total_time_seconds / 3600
total_time_days = total_time_hours / 24

print(f"\n⏱️ Data Time Span:")
print(f"  • Total seconds: {total_time_seconds:,.0f}")
print(f"  • Total hours: {total_time_hours:.1f}")
print(f"  • Total days: {total_time_days:.2f}")

# Time differences between fraud and legitimate
fraud_avg_time = time_stats.loc[1, "avg_time"]
legit_avg_time = time_stats.loc[0, "avg_time"]
time_diff = legit_avg_time - fraud_avg_time

print("\n🔍 Key Temporal Finding:")
print(f"  • Fraud average time: {fraud_avg_time:,.0f}s ({fraud_avg_time/3600:.1f} hours)")
print(f"  • Legitimate average time: {legit_avg_time:,.0f}s ({legit_avg_time/3600:.1f} hours)")
print(f"  • Difference: {time_diff:,.0f}s ({time_diff/3600:.1f} hours)")
print(f"  • Fraud occurs {time_diff/3600:.1f} hours earlier on average")


In [ ]:
# Convert Time to hours and create hour-based features
print("=" * 70)
print("CREATING TIME-BASED FEATURES")
print("=" * 70)

# Add hour-based features to dataframe
df["Time_hours"] = df["Time"] / 3600
df["Time_days"] = df["Time"] / (3600 * 24)

# Create hour bins (0 - 48 hours, binned by hour)
df["Hour_of_period"] = (df["Time"] / 3600).astype(int)

print("\n✅ Created time-based features:")
print("  • Time_hours: Continuous hours since start")
print("  • Time_days: Continuous days since start")
print("  • Hour_of_period: Hour bin (0 - 47)")

# Analyze fraud distribution by hour
fraud_by_hour_query = """
    SELECT
        CAST("Time" / 3600 as INTEGER) as hour,
        COUNT(*) as total_transactions,
        SUM(CASE WHEN "Class" = 1 THEN 1 ELSE 0 END) as fraud_count,
        SUM(CASE WHEN "Class" = 0 THEN 1 ELSE 0 END) as legit_count,
        ROUND(100.0 * SUM(CASE WHEN "Class" = 1 THEN 1 ELSE 0 END) / COUNT(*), 4) as fraud_rate_pct
    FROM fraud_data
    GROUP BY hour
    ORDER BY hour
"""

fraud_by_hour = con.execute(fraud_by_hour_query).df()

print(f"\n📊 Hourly breakdown created: {len(fraud_by_hour)} hours analyzed")
print("\n🔍 Sample - First 4 hours:")
print(fraud_by_hour.head().to_string(index=False))


In [ ]:
# Recreate class-separated dataframes with time features
legit_df = df[df["Class"] == 0].copy()
fraud_df = df[df["Class"] == 1].copy()

print("\n✅ Updated class-separated dataframes with time features")
print(f"  • Legitimate: {len(legit_df):,} rows")
print(f"  • Fraud: {len(fraud_df):,} rows" )
print(f"  • legit_df columns: {legit_df.columns}")
print(f"  • fraud_df columns: {fraud_df.columns}")
print("  • New columns available: Time_hours, Time_days, Hour_of_period")

# Visualize temporal patterns
print()
print("=" * 70)
print("VISUALIZING TEMPORAL PATTERNS")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
print(legit_df.columns)
# Plot 1: Time distribution by class
ax1 = axes[0, 0]
legit_time = legit_df["Time_hours"]
fraud_time = fraud_df["Time_hours"]

ax1.hist(legit_time, bins=48, alpha=0.6, label="Legitimate", color="green", density=True)
ax1.hist(fraud_time, bins=48, alpha=0.6, label="Fraud", color="red", density=True)
ax1.axvline(legit_time.mean(), color="darkgreen", linestyle="--", linewidth=2, label=f"Legit Mean: {legit_time.mean():.1}h")
ax1.axvline(fraud_time.mean(), color="darkred", linestyle="--", linewidth=2, label=f"Fraud Mean: {fraud_time.mean():.1}h")
ax1.set_xlabel("Time (hours)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Density", fontsize=12)
ax1.set_title("Transaction Time Distribution by Class", fontsize=13, fontweight="bold")
ax1.legend(loc="best")
ax1.grid(alpha=0.3)


# Plot 2: Fraud rate by hour
ax2 = axes[0, 1]
ax2.bar(fraud_by_hour["hour"], fraud_by_hour["fraud_rate_pct"], color="coral", alpha=0.7)
ax2.axhline(y=(df["Class"].sum() / len(df) * 100), color="red", linestyle="--", linewidth=2, 
            label=f"Overall Fraud Rate: {(df["Class"].sum() / len(df) * 100):.3f}%")
ax2.set_xlabel("Hour of Period", fontsize=12, fontweight="bold")
ax2.set_ylabel("Fraud Rate (%)", fontsize=12)
ax2.set_title("Fraud Rate by Hour", fontsize=13, fontweight="bold")
ax2.legend(loc="best")
ax2.grid(alpha=0.3, axis="y")

# Plot 3: Transaction volume by hour
ax3 = axes[1, 0]
ax3.plot(fraud_by_hour["hour"], fraud_by_hour["total_transactions"], marker="o",
         linewidth=2, markersize=4, label="Total Transactions", color="blue")
ax3.set_xlabel("Hour of Period", fontsize=12, fontweight="bold")
ax3.set_ylabel("Transaction Count", fontsize=12)
ax3.set_title("Transaction Volume Over Time", fontsize=13, fontweight="bold")
ax3.legend(loc="best")
ax3.grid(alpha=0.3)

# Plot 4: Cumulative fraud over time
ax4 = axes[1, 1]
fraud_by_hour["cumulative_fraud"] = fraud_by_hour["fraud_count"].cumsum()
fraud_by_hour["cumulative_total"] = fraud_by_hour["total_transactions"].cumsum()
fraud_by_hour["cumulative_fraud_rate"] = (fraud_by_hour["cumulative_fraud"] / fraud_by_hour["cumulative_total"]) * 100

ax4.plot(fraud_by_hour["hour"], fraud_by_hour["cumulative_fraud_rate"], linewidth=2, color="darkred")
ax4.axhline(y=(df["Class"].sum() / len(df) * 100), color="blue", linestyle="--", linewidth=2, 
            label=f"Final Fraud Rate: {(df['Class'].sum() / len(df) * 100):.3f}%")
ax4.set_xlabel("Hour of Period", fontsize=12, fontweight="bold")
ax4.set_ylabel("Cumulative Fraud Rate (%)", fontsize=13, fontweight="bold")
ax4.set_title("Cumulative Fraud Rate Over Time", fontsize=13, fontweight="bold")
ax4.legend(loc="best")
ax4.grid(alpha=0.3)

# Save the plot in local directory
plt.savefig(PLOTS_DIR / "temporal_visualization_plots.png")


plt.tight_layout()
plt.show()

print("\n✅ Temporal visualization created")


In [ ]:
# Identify high-risk time window
print("=" * 70)
print("HIGH-RISK TIME WINDOW ANALYSIS")
print("=" * 70)

# Find hours with above-average fraud rates
overall_fraud_rate = (df["Class"].sum() / len(df)) * 100
high_risk_hours = fraud_by_hour[fraud_by_hour["fraud_rate_pct"] > overall_fraud_rate].copy()

print(f"\n🚨 High-Risk Hours (fraud rate > {overall_fraud_rate:.3f}%):\n")
print(f"{'Hour':<8} {'Total Transactions':>16} {'Fraud':>8} {'Fraud Rate':>14} {'Risk Level':>12}")

print("-" * 70)

for idx, row in high_risk_hours.iterrows():
    hour = int(row["hour"])
    total = int(row["total_transactions"])
    fraud = int(row["fraud_count"])
    rate = row["fraud_rate_pct"]
    
    # Classify risk level
    if rate > overall_fraud_rate * 3:
        risk = "VERY HIGH"
    elif rate > overall_fraud_rate * 2:
        risk = "HIGH"
    else:
        risk = "ELEVATED"
    
    print(f"{hour:<8} {total:>16} {fraud:>8} {rate:>14.4f} {risk:>12}")

# Summary statistics
print(f"\n📊 High-Risk Summary:")
print(f"  • Total high-risk hours: {len(high_risk_hours)}")
print(f"  • Fraud in high-risk hours: {high_risk_hours['fraud_count'].sum()} "
      f"({(high_risk_hours['fraud_count'].sum() / df['Class'].sum()*100):.1f}% of all fraud)")
print(f"  • Peak fraud rate: {high_risk_hours['fraud_rate_pct'].max():.4f}% "
      f"(Hour {high_risk_hours.loc[high_risk_hours['fraud_rate_pct'].idxmax(), 'hour']:.0f})")
print(f"  • Lowest fraud rate: {fraud_by_hour['fraud_rate_pct'].min():.4f}% "
      f"(Hour {fraud_by_hour.loc[fraud_by_hour['fraud_rate_pct'].idxmin(), 'hour']:.0f})")


In [ ]:
# Transaction velocity analysis
print("=" * 70)
print("TRANSACTION VELOCITY ANALYSIS")
print("=" * 70)

print("\nAnalyzing transaction density and velocity patterns...")

# Calculate transactions per hour
velocity_stats = fraud_by_hour[["hour", "total_transactions", "fraud_count"]].copy()
velocity_stats["transactions_per_minute"] = velocity_stats["total_transactions"] / 60

print("\n📊 Transaction Velocity Statistics:")
print(f"  • Average transactions per hour: {velocity_stats['total_transactions'].mean():.0f}")
print(f"  • Average transactions per minute: {velocity_stats['transactions_per_minute'].mean():.2f}")
print(f"  • Peak hour volume: {velocity_stats['total_transactions'].max():.0f} transactions "
      f"(Hour {velocity_stats.loc[velocity_stats['total_transactions'].idxmax(), 'hour']:.0f})")
print(f"  • Lowest hour volume: {velocity_stats['total_transactions'].min():.0f} transactions "
     f"(Hour {velocity_stats.loc[velocity_stats['total_transactions'].idxmin(), 'hour']:.0f})")

# Identify transaction volume changes
velocity_stats['volume_change'] = velocity_stats["total_transactions"].pct_change() * 100

print("\n📈 Volume Volatility:")
print(f"  • Average volume change: {velocity_stats['volume_change'].abs().mean():.1f}%")
print(f"  • Max volume spike: {velocity_stats['volume_change'].max():.1f}% "
     f"(Hour {velocity_stats.loc[velocity_stats['volume_change'].idxmax(), 'hour']:.0f})")
print(f"  • Max volume drop: {velocity_stats['volume_change'].min():.1f}% "
     f"(Hour {velocity_stats.loc[velocity_stats['volume_change'].idxmin(), 'hour']:.0f})")

# Correlation between volume and fraud
volume_fraud_corr = velocity_stats[["total_transactions", "fraud_count"]].corr().iloc[0, 1]
print(f"\nVolume-Fraud Correlation: {volume_fraud_corr:.4f}")

if abs(volume_fraud_corr) > 0.5:
    print(f"  {'Strong' if abs(volume_fraud_corr) > 0.7 else 'Moderate'} "
          f"{'positive' if volume_fraud_corr > 0 else 'negative'} correlation")

else:
    print(f"  Weak correlation: fraud not strongly tied to volume")


In [ ]:
# Step 3 Summary
print("=" * 70)
print("STEP 3 SUMMARY: TEMPORAL PATTERN INSIGHTS")
print("=" * 70)

# Key temporal insights
fraud_early_hours = fraud_by_hour[fraud_by_hour["hour"] < 24]["fraud_count"].sum()
fraud_late_hours = fraud_by_hour[fraud_by_hour["hour"] >= 24]["fraud_count"].sum()
total_fraud = df["Class"].sum()

print(f"""
⏰ KEY TEMPORAL FINDINGS:

1. 🕐 Fraud Timing Pattern:
    • Fraud occurs {time_diff/3600:.1f} hours earlier on average
    • Fraud mean time: {fraud_avg_time/3600:.1f}h vs Legitimate: {legit_avg_time/3600:.1f}h
    • Suggests fraudsters act in first half of observation period

2. 📅 Day-by-Day Breakdown:
    • Day 1 (Hours 0-23): {fraud_early_hours} frauds ({fraud_early_hours/total_fraud*100:.1f}%)
    • Day 2 (Hours 24-47): {fraud_late_hours} frauds ({fraud_late_hours/total_fraud*100:.1f}%)
    • Fraud is {'more concentrated in Day 1' if fraud_early_hours > fraud_late_hours
    else 'more concentrated in Day 2' if fraud_late_hours > fraud_early_hours else 'evenly distributed'}

3. 🚨 High-Risk Time Windows:
    • {len(high_risk_hours)} hours have above-average fraud rates
    • Peak fraud rate: {high_risk_hours['fraud_rate_pct'].max():.4f}% 
    (Hour {high_risk_hours.loc[high_risk_hours['fraud_rate_pct'].idxmax(), 'hour']:.0f})
    • High-risk hours contain {high_risk_hours['fraud_count'].sum()} frauds
    ({high_risk_hours['fraud_count'].sum()/total_fraud*100:.1f}% of all fraud)

4. ⚡ Transaction Velocity:
    • Average: {velocity_stats['total_transactions'].mean():.0f} transactions/hour
    • Volume-fraud correlation: {volume_fraud_corr:.4f} ({'weak' if abs(volume_fraud_corr) < 0.5 
    else 'moderate' if abs(volume_fraud_corr) < 0.7 else 'strong'})
    • {'Fraud increases with volume' if volume_fraud_corr > 0.3 else 'Fraud not strongly tied to transaction volume'}

💡 MODELING IMPLICATIONS:

✅ Feature Engineering Opportunities:
  1. Hour_of_period (0-48) - categorical or binned feature
  2. Is_day_1 vs Is_day_2 - binary flag
  3. Is_early_period - binary flag
  4. Is_high_risk_hour - binary flag based on historical fraud rate
  5. Time_from_start - continuous hours since period start
  6. Transaction_velocity_window - rolling transaction count
  7. Cumulative_transaction_count - position in sequence

✅ Deployment Considerations:
  • Enhanced monitoring during high-risk hours (Hours {', '.join(high_risk_hours.nsmallest(3, 'hour')['hour'].astype(int).astype(str))})
  • Real-time fraud scoring should weight time features
  • Consider time-of-day adjustments to decision thresholds

⚠️ Important Note:
  • Dataset spans only 2 days - patterns may not generalize to day-of-week
  • Production system should track temporal drift over longer periods
  • Hour-of-day patterns in real deployment may differ
""")

print("\nStep 3 complete: Temporal patterns documented")
print("  Ready to investivgate duplicate transactions")
